In [ ]:
########## __functions.py ##########

In [9]:
# Import (from __functions.py)
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from shapely import Polygon, wkt

In [11]:
# CSV to GeoDataFrame - imports a CSV and converts it to a GeoDataFrame
#     assumes geometry column is called 'geometry'
def csv_gdf(fp, crs):
    df = pd.read_csv(fp, parse_dates=['datetime'])  # open CSV
    df['geometry'] = df['geometry'].apply(wkt.loads)  # load geometries
    gdf = gpd.GeoDataFrame(df, geometry='geometry', crs=crs)  # convert to GeoDataFrame
    return gdf


In [12]:
# Gridder - creates a grid over a given area based on the set height, width, and CRS
def gridder(area, h, w, crs):
    area = area.dissolve().to_crs(crs)  # dissolve and reproject area to desired CRS
    xmin, ymin, xmax, ymax = area.total_bounds  # get total bounds of the area
    xs = list(np.arange(xmin, xmax + w, w))  # create list of x values
    ys = list(np.arange(ymin, ymax + h, h))  # create list of y values
    polygons = []  # list for polygons
    for x in xs[:-1]:
        for y in ys[:-1]:
            polygons.append(Polygon([  # create and append polygon
                (x, y),  # bottom left
                (x + w, y),  # bottom right
                (x + w, y + h),  # top right
                (x, y + h)]))  # top left
    g = gpd.GeoDataFrame({'geometry': polygons}, crs=crs)  # create the grid GeoDataFrame
    g = g[g.centroid.intersects(area.geometry[0])]  # keep only grid cells whose centroids are in area
    # g = g[g.intersects(area.geometry[0])]  # keep only grid cells that are at least partially in the area
    return g


In [13]:
# Grid plotter - plots a grid
def grid_plotter(b, g):
    base = b.plot(alpha=0.2)
    g.plot(ax=base, facecolor='none', edgecolor='#707070')
    plt.show()


In [ ]:
########## __setup.py ##########

In [19]:
# Import (from __setup.py)
import geopandas as gpd
import glob
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import rasterstats
import rioxarray as rxr
import shapely
from shapely import wkt
import xarray as xr

In [20]:
#####
data_folder = '~/DansData/BOF/'  # set the data folder
crs_local = 32619  # set the CRS

In [21]:
#####
# Temporal
tm_years = (2005, 2006)  # set the year range
tm_months = (8, 10)  # set the month range
tm_season = 'month'  # set the season
tm_name = (str(tm_years[0]) + '-' + str(tm_years[1]) +
           '_x_' + str(tm_months[0]) + '-' + str(tm_months[1]) +
           '_x_' + tm_season)
tm_name

'2005-2006_x_8-10_x_month'

In [24]:
#####
# Spatial
sp_extent_name = 'bof'  # set the spatial extent name
sp_extent = gpd.read_file(data_folder + sp_extent_name + '.gpkg').to_crs(crs_local)  # input the spatial extent file
sp_size = (10000, 10000)  # set the size (height, width) of spatial samples (i.e., grid cells)
sp_name = sp_extent_name + '_' + str(sp_size[0]) + 'x' + str(sp_size[1])  # the name of the spatial extent x size combination
sp_name

'bof_10000x10000'

In [ ]:
########## a_sptm.py ##########

from BOF_MSOM.BOF_sptm.__setup import *
from BOF_MSOM.BOF_sptm.__functions import *


# Load survey effort
se = pd.read_csv(data_folder + 'BOF_MSOM/BOF_sesienv/g_geometries/BOF_ON_se_geoms.csv', parse_dates=['datetime_beg', 'datetime_end'])

#####
# Temporal

# Apply temporal extents
se = se.loc[(se['year'] >= tm_years[0]) & (se['year'] <= tm_years[1])]
se = se.loc[(se['month'] >= tm_months[0]) & (se['month'] <= tm_months[1])]

# Create labels
yr_labels = ['y' + str(y) for y in se['year'].unique()]
sn_labels = se['sn_id'].unique().tolist()
sv_labels = se['sv_id'].unique().tolist()

#####
# Spatial
grid = gridder(sp_extent, sp_size[0], sp_size[1], crs_local)  # create grid

se['geometry'] = se['geometry'].apply(wkt.loads)  # load geometries of the SE tracklines
se = gpd.GeoDataFrame(se, geometry='geometry', crs=4326).to_crs(crs_local)  # convert SE to GeoDataFrame
grid = grid.copy()[grid.intersects(se.dissolve().geometry.iloc[0])]  # keep only grid cells that intersect with the SE

grid.reset_index(inplace=True, drop=True)  # reset index and drop former index column as index is no longer continuous as certain cells were removed
grid.reset_index(inplace=True)  # reset index again...
grid.rename(columns={'index': 'sp_id'}, inplace=True)  # ...and use second former index to...
grid['sp_id'] = grid['sp_id'].apply(lambda gc: 'g' + str(gc).zfill(5))  # ...create spatial IDs for each grid cell, e.g., g00024

base = sp_extent.plot(alpha=0.2)
grid.plot(ax=base, facecolor='none', edgecolor='#707070')
se.plot(ax=base)

grid.to_file(data_folder + 'BOF_MSOM/BOF_sptm/a_sptm/' + sp_name + '.gpkg')  # output to GeoPackage

#####
# Spatiotemporal template
sptm = pd.DataFrame(grid['sp_id'], columns=['sp_id'])  # create a dataframe with the spatial IDs
sptm['year'] = [yr_labels for i in sptm.index]  # add the temporal IDs as a list to each row then...
sptm = sptm.assign(year=sptm['year']).explode('year')  # ...explode so that the rows contain all combinations of spatial and temporal IDs, i.e., all spatiotemporal periods
sptm['sn_id'] = [sn_labels for i in sptm.index]  # add the temporal IDs as a list to each row then...
sptm = sptm.assign(sn_id=sptm['sn_id']).explode('sn_id')  # ...explode so that the rows contain all combinations of spatial and temporal IDs, i.e., all spatiotemporal periods
sptm['sv_id'] = [sv_labels for j in sptm.index]  # add the temporal IDs as a list to each row then...
sptm = sptm.assign(sv_id=sptm['sv_id']).explode('sv_id')  # ...explode so that the rows contain all combinations of spatial and temporal IDs, i.e., all spatiotemporal periods
sptm.to_csv(data_folder + 'BOF_MSOM/BOF_sptm/a_sptm/' + sp_name + '__' + tm_name + '.csv', index=False)  # output to CSV


In [17]:
%reset -f

In [18]:
whos

Interactive namespace is empty.
